# CircuitSight — Fine-Tuning & Evaluation (Qwen2.5-VL-3B, QLoRA)

Trains a small vision-language model to **read a circuit schematic and solve it** (components → topology → equations → values), using the dataset produced by `CircuitSight_dataset_generation.ipynb`.

**Run order:**
1. Setup + GPU check
2. Load the dataset (from Drive or an uploaded zip)
3. Load Qwen2.5-VL-3B (4-bit) + LoRA
4. **Day-1 OOM smoke test — the go/no-go gate.** Run this *before* committing to a full run. If it OOMs, follow the fallback notes (lower resolution or smaller model) before proceeding.
5. Full training
6. Inference
7. **Evaluation harness** — scores base vs. tuned on: component accuracy, R_eq accuracy, final-answer accuracy, and — separately — **fabrication rate** vs. **honest-abstention rate**.

Needs a GPU runtime (Runtime → Change runtime type → GPU). With Colab credits, an **A100 40GB** is comfortable for the 3B; a T4/L4 works at lower resolution or with SmolVLM.


## 1. Setup

In [ ]:
# Install a torch version Unsloth supports (let pip choose the right CUDA build).
!pip install -q "torch==2.6.0" "torchvision==0.21.0"
!pip install -q unsloth sympy
print("installed — RESTART SESSION before importing")

In [ ]:
import torch; print("torch", torch.__version__)
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
print("unsloth ready")

## 2. Config

In [ ]:
CFG = dict(
    MODEL       = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",  # fallback: SmolVLM / a 2B if VRAM is tight
    MAX_IMAGE_PX= 512,     # longest image side; the main VRAM + speed knob. 512 ~= 2x faster than 768 on a T4.
    LORA_R      = 16, LORA_ALPHA = 16,
    BATCH       = 2, GRAD_ACCUM = 4,   # effective batch = BATCH*GRAD_ACCUM = 8. Bigger BATCH = faster but more VRAM; run the §4 smoke test first. OOM? -> BATCH=1, GRAD_ACCUM=8 (same effective 8, no extra VRAM). A100: BATCH 4-8 ok.
    MAX_STEPS   = 350,     # <-- caps optimizer steps (~30-40 min on a T4). Set to None to train a full epoch.
    EPOCHS      = 1,       # only used when MAX_STEPS is None
    LR          = 2e-4,
    MAX_LEN     = 2048,
    SEED        = 3407,
    DATA_DIR    = "circuitsight_dataset",   # unzipped dataset folder
    SAVE_TO_DRIVE = False,   # if True, snapshot the trained adapter to DRIVE_DIR/models (keep 2 most recent)
    DRIVE_DIR   = "/content/drive/MyDrive/CircuitSight",   # your project folder in Google Drive
    OUT_DIR     = "circuitsight_qlora",
)
# Why this many steps? EPOCHS=1 over ~10k images at effective batch BATCH*GRAD_ACCUM=4 is ~2,600
# optimizer steps (~6h on a T4) -- that is steps, not epochs. This narrow, structured behavior is
# learned in a few hundred steps, so we cap with MAX_STEPS. 350 steps ~= 2,800 images seen (effective batch 8); the
# trainer shuffles the full dataset, so all families + value-modes (numeric/symbolic/mixed) are
# sampled in proportion to the dataset mix (see CONFIG fractions). Watch the section-9 eval curve;
# raise MAX_STEPS (or set None) only if it's still climbing.
INSTRUCTION = ("You are a circuit analysis tutor. Look at the schematic and answer the question. "
    "First list every component, each with the image region it occupies as "
    "<box>[x0,y0,x1,y1]</box> in 0-1000 normalized coordinates. Then state the concepts used and "
    "the topology (what is in series/parallel). If there is a capacitor or inductor, apply its "
    "steady-state / t=0 behavior (a capacitor is open at steady state and a wire at t=0; an "
    "inductor is the reverse). Component values may be numbers (e.g. 100Ω, 12V) or symbols (e.g. "
    "R1, R2, V) — if they are symbols, give the answer as an algebraic expression in those symbols. "
    "Solve step by step, showing intermediate values (e.g. R_eq and each branch current), and "
    "end with a self-check. If a component value is not legible, say so and report the answer as "
    "null instead of guessing. End with a single line "
    "'FINAL: {\"quantity\":..., \"target_id\":..., \"value\":..., \"unit\":..., \"abstain\":false}'.")
CFG

## 3. Load the dataset

Get the dataset next to this notebook. Either mount Drive and point `DATA_DIR` at the unzipped folder, or upload the zip produced by the data-gen notebook.

In [ ]:
import os, json, zipfile
from collections import Counter

# ---- Load the dataset from Google Drive ----
DATASET_ZIP = "/content/drive/MyDrive/slm_project/data/circuitsight_dataset_20260710_1.zip"  # <- your zip

# mount Drive if it isn't already
if DATASET_ZIP.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")

# sanity: the file exists and is a real (non-truncated) zip -> clear message instead of a raw BadZipFile
assert os.path.exists(DATASET_ZIP), f"{DATASET_ZIP} not found - check the path and that Drive is mounted."
with open(DATASET_ZIP, "rb") as _f:
    assert _f.read(4) == b"PK\x03\x04", f"{DATASET_ZIP} is not a valid .zip (truncated upload or wrong file)."

# the dataset was zipped from INSIDE the dataset folder, so its contents (images/, train.jsonl, ...)
# sit at the zip root -> extract into CFG["DATA_DIR"].
os.makedirs(CFG["DATA_DIR"], exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP) as z:
    z.extractall(CFG["DATA_DIR"])
print("unzipped", DATASET_ZIP, "->", CFG["DATA_DIR"])

IMG_DIR = os.path.join(CFG["DATA_DIR"], "images")
assert os.path.isdir(IMG_DIR), f"expected {IMG_DIR} after unzip - check the zip's internal structure"

def load_jsonl(name):
    p = os.path.join(CFG["DATA_DIR"], name)
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []

train_rows = load_jsonl("train.jsonl")
val_rows   = load_jsonl("val_synthetic.jsonl")
print(f"train: {len(train_rows)}  val: {len(val_rows)}")
print("example keys:", list(train_rows[0].keys()))
print("value_mode mix:", Counter(r.get("value_mode","MISSING->stale zip!") for r in train_rows))
# sanity: confirm this is the PLAN-retrain dataset (resistor targets should carry a PLAN line)
_has_plan = sum("\nPLAN:" in r.get("target_output","") for r in train_rows)
print(f"targets with a PLAN line: {_has_plan}/{len(train_rows)}  (0 => generated with INCLUDE_PLAN=False)")

# --- alternative: upload the zip to /content instead of Drive ---
# from google.colab import files; up = files.upload(); DATASET_ZIP = "/content/"+next(iter(up))


## 4. Format as vision chat samples

Each row → a user turn (image + instruction + question) and an assistant turn (the worked solution). Images are loaded as RGB PIL and downsized to `MAX_IMAGE_PX` — passing real PIL images (not paths) avoids the common *'could not make a flat list of images'* collator error.

In [ ]:
# All imports the training + eval cells rely on (safe to re-run anytime).
import os, json, zipfile, re, random, time
import torch
import numpy as np
from PIL import Image, ImageDraw
from datasets import Dataset
from collections import Counter

# unsloth / trl (already installed; just binding the names in this session)
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

print("all imports loaded — PIL, torch, datasets, unsloth, trl ready")

In [ ]:
# Lazy image loading: keep only lightweight text rows in RAM; decode each image
# on demand when the trainer pulls it. This avoids holding 11k decoded images at once.
def load_image(name):
    img = Image.open(os.path.join(IMG_DIR, name)).convert("RGB")
    m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m / max(img.size)
        img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    return img

def to_conversation(row):
    return {"messages": [
        {"role":"user","content":[
            {"type":"image","image": load_image(row["image"])},
            {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + row["question"]}]},
        {"role":"assistant","content":[{"type":"text","text": row["target_output"]}]},
    ]}

class LazyConvDataset:
    """Builds each conversation (and loads its image) only when accessed."""
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        if isinstance(i, slice):
            return LazyConvDataset(self.rows[i])   # slicing -> a smaller lazy dataset
        return to_conversation(self.rows[i])
    def select(self, idxs):                        # HF-style helper, used by some trainers
        return LazyConvDataset([self.rows[k] for k in idxs])

train_conv = LazyConvDataset(train_rows)
print("lazy dataset ready:", len(train_conv), "samples (images load on demand)")
print("sample user text:\n", train_conv[0]["messages"][0]["content"][1]["text"][:200])

# def load_image(name):
#     img = Image.open(os.path.join(IMG_DIR, name)).convert("RGB")
#     m = CFG["MAX_IMAGE_PX"]
#     if max(img.size) > m:
#         s = m / max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
#     return img

# def to_conversation(row):
#     return {"messages": [
#         {"role":"user","content":[
#             {"type":"image","image": load_image(row["image"])},
#             {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + row["question"]}]},
#         {"role":"assistant","content":[{"type":"text","text": row["target_output"]}]},
#     ]}

# train_conv = [to_conversation(r) for r in train_rows]
# print("formatted", len(train_conv), "samples")
# print("sample user text:\n", train_conv[0]["messages"][0]["content"][1]["text"][:200])

## 5. Load model + attach LoRA

We finetune both vision and language layers: component *identification* is a vision-side skill, equation *setup* is language-side, and this task needs both.

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    CFG["MODEL"], load_in_4bit=True, use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=CFG["LORA_R"], lora_alpha=CFG["LORA_ALPHA"], lora_dropout=0,
    bias="none", random_state=CFG["SEED"])
print("model + LoRA ready")

## 6. Day-1 OOM smoke test — the go/no-go gate

**Run this before the full training.** It trains 5 steps on a handful of samples and reports peak VRAM. Purpose: find out *today* whether the model trains without OOM at your image resolution, instead of discovering it 3 days in.

- **Passes** (completes, peak VRAM leaves headroom) → proceed to full training.
- **OOMs** → in order: (1) drop `MAX_IMAGE_PX` to 512 and re-run cell 4; (2) set `finetune_vision_layers=False`; (3) switch `CFG["MODEL"]` to a 2B (e.g. SmolVLM) and reload cell 5. Re-run this gate until it passes.


In [ ]:
import time
FastVisionModel.for_training(model)
smoke = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=LazyConvDataset(train_rows[:8]),
    args=SFTConfig(
        per_device_train_batch_size=CFG["BATCH"], gradient_accumulation_steps=1,
        warmup_steps=0, max_steps=5, learning_rate=CFG["LR"], logging_steps=1,
        optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
        seed=CFG["SEED"], output_dir="smoke_out", report_to="none",
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True}, max_length=CFG["MAX_LEN"],
    ))
torch.cuda.reset_peak_memory_stats()
t=time.time(); smoke.train(); dt=time.time()-t
peak = torch.cuda.max_memory_reserved()/1e9
total = torch.cuda.get_device_properties(0).total_memory/1e9
print(f"\nSMOKE TEST PASSED: 5 steps in {dt:.0f}s | peak VRAM {peak:.1f} / {total:.1f} GB")
print("GO." if peak < 0.9*total else "TIGHT - lower MAX_IMAGE_PX before the full run.")

## 7. Full training

**This is capped by `CFG["MAX_STEPS"]`, not epochs.** A full epoch over ~10k images at effective batch 4 is ~2,600 optimizer steps (≈6 h on a T4) — that's *steps*, not epochs. This behavior is highly structured and narrow, so a few hundred steps is plenty. `MAX_STEPS=150` at `MAX_IMAGE_PX=512` runs in roughly **15–25 min on a T4**. The trainer shuffles the full dataset, so 150 steps still samples across all three families.

**Scale strategy:** run 150 steps, look at the section-9 eval curve, and only raise `MAX_STEPS` (or set it to `None` for a full epoch) if accuracy is still climbing — extra steps past diminishing returns just burn time. If you OOM or it's too slow, keep `MAX_IMAGE_PX=512`; if you have headroom and want sharper boxes, try 640/768.

In [ ]:
FastVisionModel.for_training(model)
eff_batch = CFG["BATCH"]*CFG["GRAD_ACCUM"]
use_steps = CFG.get("MAX_STEPS")
schedule  = dict(max_steps=use_steps) if use_steps else dict(num_train_epochs=CFG["EPOCHS"])
seen      = use_steps*eff_batch if use_steps else len(train_conv)
print(f"training: {'max_steps='+str(use_steps) if use_steps else 'epochs='+str(CFG['EPOCHS'])}"
      f" | effective batch {eff_batch} | ~{seen} images seen (dataset has {len(train_conv)}, shuffled)")

# periodic checkpoints so a disconnect does not lose the run (keeps the 2 most recent)
if CFG.get("SAVE_TO_DRIVE"):
    from google.colab import drive; drive.mount("/content/drive")
    CKPT_DIR = os.path.join(CFG["DRIVE_DIR"], "checkpoints")   # written straight to Drive during training
else:
    CKPT_DIR = CFG["OUT_DIR"]                                    # /content/circuitsight_qlora (download manually)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"checkpoints -> {CKPT_DIR}  (every 100 steps, keeping the 2 most recent)")

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_conv,
    args=SFTConfig(
        per_device_train_batch_size=CFG["BATCH"], gradient_accumulation_steps=CFG["GRAD_ACCUM"],
        warmup_steps=5, learning_rate=CFG["LR"],
        logging_steps=10, optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
        seed=CFG["SEED"], output_dir=CKPT_DIR, report_to="none",
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True}, max_length=CFG["MAX_LEN"],
        save_strategy="steps", save_steps=100, save_total_limit=2,   # mid-run checkpoints for disconnect safety (keep 2 newest)
        **schedule,
    ))
stats = trainer.train()
print("done. final loss:", round(stats.training_loss, 4))

## 8. Inference helper

In [ ]:
def solve_image(pil_img, question, model, tokenizer, max_new_tokens=896):
    FastVisionModel.for_inference(model)
    msgs = [{"role":"user","content":[{"type":"image"},
             {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + question}]}]
    text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = tokenizer(pil_img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True,
                         do_sample=False, temperature=0.0)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip().split("\nPLAN:")[0].strip()

# quick look
r0 = val_rows[0] if val_rows else train_rows[0]
print("Q:", r0["question"])
print(solve_image(load_image(r0["image"]), r0["question"], model, tokenizer)[:500])

## 9. Evaluation harness (base vs. tuned)

This is the point of the whole project. The harness parses each model output (preferring the machine-readable `FINAL: {quantity, target_id, value, unit, abstain}` line, with a prose fallback) and scores it against the exact gold labels along separate axes:

- **component_accuracy** — right resistor count *and* the right component **types** present (R vs C vs L).
- **reactive_type_accuracy** — on capacitor/inductor circuits, did it identify the reactive component as the correct type (not confuse a capacitor for an inductor)? This is the headline **"spatial blindness"** number.
- **Req_accuracy / answer_accuracy / step_verified_accuracy** — did it get the final quantity right, *and* the `R_eq` intermediate right (a correct answer from a wrong intermediate does **not** count).
- **grounding_accuracy** (IoU ≥ 0.5 vs. gold boxes) and **grounding_hallucination_rate** — did the cited regions actually land on the components, or invent locations?
- **fabrication_rate** vs. **honest_abstention_rate** — on illegible-value cases, did it invent a number (bad) or correctly say null (good)?
- **concept_hallucination_rate** — did it invoke a solving concept the problem doesn't need?
- **answer_accuracy_by_qtype** — a breakdown across current / voltage / power / resistance / charge / energy.

We run it on the **base** model and the **tuned** model over the same held-out set and compare. Report the **real-world** transfer set (§10) as the headline number.


### Load the model to evaluate — from Google Drive

Run this **before** the §9 harness / §9 run / §9d when you're in a fresh runtime (e.g. the session ended after training). It mounts Drive, loads the fine-tuned adapter from `MODEL_ZIP`, and (re)defines `load_image`/`solve_image`, so the eval cells work without re-running §4/§8. **Skip it** if you just trained in this session (the model is already in memory).

Set `MODEL_ZIP` to your zip in `MyDrive/slm_project/models/`.


In [ ]:
# ---- Load the model to evaluate, from Google Drive (run before §9 / §9d in a fresh runtime) ----
# Skip if you just trained in this session (the model is already in memory).
MODEL_ZIP = "/content/drive/MyDrive/slm_project/models/circuitsight_qlora_final_20260710_1.zip"  # <- your model zip

import os, glob, zipfile
from unsloth import FastVisionModel
from PIL import Image

# mount Drive if needed
if MODEL_ZIP.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")

# locate + integrity-check the zip (clear message instead of a raw BadZipFile)
if not os.path.exists(MODEL_ZIP):
    print("Not found. Zips in that folder:")
    for z in sorted(glob.glob(os.path.join(os.path.dirname(MODEL_ZIP), "*.zip"))): print("  ", z)
    raise FileNotFoundError(MODEL_ZIP)
with open(MODEL_ZIP, "rb") as _f:
    assert _f.read(4) == b"PK\x03\x04", f"{MODEL_ZIP} is not a valid .zip (truncated / wrong file)."

# unzip + load the adapter (finds adapter_config.json at any depth)
UNZIP = "/content/loaded_model"; os.makedirs(UNZIP, exist_ok=True)
with zipfile.ZipFile(MODEL_ZIP) as z: z.extractall(UNZIP)
ADAPTER_DIR = os.path.dirname(glob.glob(os.path.join(UNZIP, "**", "adapter_config.json"), recursive=True)[0])
print("adapter at:", ADAPTER_DIR)
model, tokenizer = FastVisionModel.from_pretrained(ADAPTER_DIR, load_in_4bit=True,
                                                   use_gradient_checkpointing="unsloth")
FastVisionModel.for_inference(model); print("loaded model from", os.path.basename(MODEL_ZIP))

# image loader + solver (so §9 / §9d run even if §4 and §8 weren't run this session)
def load_image(name):
    img = Image.open(os.path.join(IMG_DIR, name)).convert("RGB"); m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m/max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    return img
def solve_image(pil_img, question, model, tokenizer, max_new_tokens=896):
    FastVisionModel.for_inference(model)
    msgs=[{"role":"user","content":[{"type":"image"},
           {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + question}]}]
    text=tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs=tokenizer(pil_img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out=model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0, use_cache=True)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip().split("\nPLAN:")[0].strip()
print("model, tokenizer, load_image, solve_image ready -> run the §9 harness, then §9 / §9d")


In [ ]:
import re, json, random
import sympy as smp
# ============================================================================
# Eval harness for the generalized schema:
#   FINAL: {quantity, target_id, value, unit, abstain}
# Scores, separately, the axes the BrainLift cares about:
#   component identification (count + TYPE: R vs C vs L), grounding (IoU>=0.5) +
#   grounding-hallucination, step-verified solve (final AND R_eq intermediate),
#   fabrication vs honest-abstention, concept-hallucination, and — the headline
#   "spatial blindness" number — reactive component-type accuracy (cap vs inductor).
# ============================================================================
def parse_final(text):
    m = re.search(r"FINAL:\s*(\{.*?\})", text)   # non-greedy, no DOTALL: a trailing PLAN line must NOT be swallowed
    if m:
        try:
            j = json.loads(m.group(1))
            return {"quantity":j.get("quantity"),"target_id":j.get("target_id"),
                    "value":j.get("value"),"unit":j.get("unit"),"abstain":bool(j.get("abstain",False))}
        except Exception: pass
    low = text.lower()                                     # prose fallback
    ab = ("null" in low) or ("not legible" in low) or ("cannot" in low)
    val=None
    m2 = re.search(r"answer[^=]*=\s*[^=]*?(-?\d+(?:\.\d+)?(?:e-?\d+)?)", low)
    if m2:
        try: val=float(m2.group(1))
        except Exception: val=None
    return {"quantity":None,"target_id":None,"value":val,"unit":None,"abstain":ab}

def parse_components(text):
    res_ids = set(re.findall(r"\bR(\d+)\b", text))
    has_cap = bool(re.search(r"capacitor|\u00b5F|uF|\bC1\b", text, re.I))
    has_ind = bool(re.search(r"inductor|\bmH\b|\bL1\b", text, re.I))
    m = re.search(r"(\d+)\s*resistor", text.lower())
    return {"n_res": int(m.group(1)) if m else len(res_ids), "has_cap":has_cap, "has_ind":has_ind}

def parse_boxes(text):
    pat=r"(V\d+|R\w+|C\w+|L\w+|E\w*|ε\w*|SW|VM)\s*<box>\s*\[?\s*(-?\d+)\s*,\s*(-?\d+)\s*,\s*(-?\d+)\s*,\s*(-?\d+)\s*\]?\s*</box>"
    return {cid:[int(a),int(b),int(c),int(d)] for cid,a,b,c,d in re.findall(pat, text)}

def _iou(p,g):
    ix0,iy0,ix1,iy1=max(p[0],g[0]),max(p[1],g[1]),min(p[2],g[2]),min(p[3],g[3])
    inter=max(0,ix1-ix0)*max(0,iy1-iy0)
    u=max(0,p[2]-p[0])*max(0,p[3]-p[1])+max(0,g[2]-g[0])*max(0,g[3]-g[1])-inter
    return inter/u if u>0 else 0.0

_CKEYS={"ohm":"ohm","parallel":"parallel","series":"series","capacitor":"capacitor","inductor":"inductor"}
def _concepts_text(text):
    m=re.search(r"concepts used:\s*(.+)", text.lower())
    if not m: return None
    chunk=m.group(1).split("\n")[0]
    return {v for k,v in _CKEYS.items() if k in chunk}
def _concepts_gold(concepts):
    s=" ".join(concepts).lower(); return {v for k,v in _CKEYS.items() if k in s}

_NUM=re.compile(r"-?\d+\.?\d*(?:[eE][+-]?\d+)?$")
_SYM_RNG=random.Random(12345)   # fixed seed -> reproducible symbolic-equivalence checks
def _is_num(x):
    return isinstance(x,(int,float)) or (isinstance(x,str) and bool(_NUM.match(x.strip())))
def sym_equal(a, b, trials=6):
    """Algebraic equivalence via random substitution: plug the same random reals into the
    shared free symbols and compare (robust to any equivalent form, e.g. R1+R2 == R2+R1)."""
    try:
        ea=smp.sympify(str(a)); eb=smp.sympify(str(b))
    except Exception:
        return str(a).replace(" ","")==str(b).replace(" ","")
    syms=sorted(ea.free_symbols | eb.free_symbols, key=str)
    if not syms:
        try: return abs(float(ea)-float(eb))<=1e-6*max(1.0,abs(float(eb)))
        except Exception:
            try: return smp.simplify(ea-eb)==0
            except Exception: return False
    for _ in range(trials):
        subsd={s:_SYM_RNG.uniform(1.5,9.5) for s in syms}
        try: fa=float(ea.subs(subsd)); fb=float(eb.subs(subsd))
        except Exception:
            try: return smp.simplify(ea-eb)==0
            except Exception: return False
        if abs(fa-fb) > 1e-6*max(1.0,abs(fb)): return False
    return True
def _close(pred, gold, rel=0.03):
    """Numeric (3% tol) when both numeric; else algebraic equivalence (symbolic answers)."""
    if pred is None or gold is None: return False
    if _is_num(pred) and _is_num(gold):
        return abs(float(pred)-float(gold)) <= max(1e-9, abs(float(gold))*rel)
    return sym_equal(pred, gold)

def score_record(gold, model_text, rel_tol=0.03, ground_mode="id"):
    ga=bool(gold["abstain"]); fam=gold.get("family","dc_resistor")
    p=parse_final(model_text); comp=parse_components(model_text); mb=parse_boxes(model_text)
    gc=gold["gold_components"]
    res={"family":fam,"question_type":gold.get("question_type"),"value_mode":gold.get("value_mode","numeric"),
         "abstain_gold":ga,"abstain_pred":bool(p["abstain"]),
         "comp_ok":None,"react_type_ok":None,"req_ok":None,"answer_ok":None,"intermediates_ok":None,
         "fabricated":None,"honest_abstain":None,
         "concept_declared":None,"concept_hallucinated":None,
         "grounding_ok":None,"grounding_hallucinated":None}
    # component identification: right resistor COUNT and right presence of cap/inductor TYPE
    res["comp_ok"]=(comp["n_res"]==gc.get("resistor")) and \
                   (comp["has_cap"]==("capacitor" in gc)) and (comp["has_ind"]==("inductor" in gc))
    if fam=="reactive":                                    # cap-vs-inductor confusion (headline)
        want_cap="capacitor" in gc
        res["react_type_ok"]=bool((comp["has_cap"] and not comp["has_ind"]) if want_cap
                                  else (comp["has_ind"] and not comp["has_cap"]))
    # grounding: fraction of gold components boxed with IoU>=0.5; hallucination = boxes that miss
    gb=gold.get("gold_boxes",{})   # ground_mode="iou": id-agnostic greedy match (real images number their own way)
    if gb:
        if ground_mode=="iou":
            preds=list(mb.values()); used=[False]*len(preds); hit=0
            for g in gb.values():
                best=0.0; bi=-1
                for i,pbx in enumerate(preds):
                    if used[i]: continue
                    v=_iou(pbx,g)
                    if v>best: best=v; bi=i
                if bi>=0 and best>=0.5: used[bi]=True; hit+=1
            res["grounding_ok"]=hit/len(gb)
            if preds: res["grounding_hallucinated"]=sum(1 for u in used if not u)/len(preds)
        else:
            res["grounding_ok"]=sum(1 for cid,g in gb.items() if cid in mb and _iou(mb[cid],g)>=0.5)/len(gb)
            if mb:
                res["grounding_hallucinated"]=sum(1 for cid,m in mb.items()
                                                  if cid not in gb or _iou(m,gb[cid])<0.5)/len(mb)
    # concept scoping
    gset=_concepts_gold(gold.get("concepts",[])); pc=_concepts_text(model_text)
    if pc is None: res["concept_declared"]=False
    else: res["concept_declared"]=True; res["concept_hallucinated"]=len(pc-gset)>0
    # answer / abstention
    if ga:
        gave=(p["value"] is not None) and (not p["abstain"])
        res["fabricated"]=gave; res["honest_abstain"]=bool(p["abstain"]) and not gave
    else:
        ga_dict=gold.get("gold_answer") or {}          # perception-only records have gold_answer=None
        gv=ga_dict.get("value")                        # float | expr-string | None
        gvals=gold.get("gold_values") or {}
        res["answer_ok"]=_close(p["value"], gv, rel_tol) if gv is not None else None
        qt=gold.get("question_type")
        if qt=="resistance":
            res["req_ok"]=res["answer_ok"]
        elif qt in ("topology","components") or gv is None:   # perception-only: no numeric intermediate
            res["req_ok"]=None
        elif gvals.get("I_total")==0:                  # open circuit: no loop-R_eq intermediate
            res["req_ok"]=None
        elif "R_eq" in gvals:
            gold_req=gvals["R_eq"]
            if _is_num(gold_req):
                mR=re.search(r"r_eq\s*=\s*(-?\d+\.?\d*(?:[eE][+-]?\d+)?)", model_text, re.I)
                predR=mR.group(1) if mR else None
            else:
                mR=re.search(r"r_eq\s*=\s*([^\n]+?)\s*(?:ohm|\u03a9|\(=|$)", model_text, re.I) or \
                   re.search(r"r_eq\s*=\s*([^\n.]+)", model_text, re.I)
                predR=mR.group(1).strip().rstrip(".") if mR else None
            res["req_ok"]=_close(predR, gold_req, rel_tol)
        gbc=gvals.get("branch_currents",{})            # full intermediate verification
        if gbc:
            pbc={cid:v.strip() for cid,v in re.findall(r"I\((R\w+)\)\s*=\s*([^,\n]+?)\s*A", model_text)}
            checked=ok=0
            for cid,gvv in gbc.items():
                if _is_num(gvv) and abs(float(gvv))<1e-3: continue   # skip sub-mA (numeric rounding noise)
                checked+=1; pv=pbc.get(cid)
                if pv is not None and _close(pv, gvv, rel_tol): ok+=1
            res["intermediates_ok"]=(ok==checked) if checked else None
    return res

def aggregate(results):
    non=[r for r in results if not r["abstain_gold"]]; ab=[r for r in results if r["abstain_gold"]]
    rj =[r for r in results if r["family"]=="reactive"]
    def frac(xs,k):
        xs=[r[k] for r in xs if r[k] is not None]; return round(sum(xs)/len(xs),4) if xs else None
    step=[r for r in non if r["comp_ok"] and (r["req_ok"] in (True,None)) and (r["intermediates_ok"] in (True,None)) and r["answer_ok"]]
    qts=sorted(set(r["question_type"] for r in non if r["question_type"]))
    _gr=frac(results,"grounding_ok"); _gh=frac(results,"grounding_hallucinated")
    _gp=(round(1-_gh,4) if _gh is not None else None)
    _gf1=(round(2*_gp*_gr/(_gp+_gr),4) if (_gp and _gr and _gp+_gr>0) else None)
    _tp=sum(1 for r in results if r["abstain_gold"] and r["abstain_pred"])
    _fp=sum(1 for r in results if (not r["abstain_gold"]) and r["abstain_pred"])
    _fn=sum(1 for r in results if r["abstain_gold"] and not r["abstain_pred"])
    _apr=(round(_tp/(_tp+_fp),4) if _tp+_fp else None); _arc=(round(_tp/(_tp+_fn),4) if _tp+_fn else None)
    return {"n_total":len(results),"n_abstain":len(ab),"n_reactive":len(rj),
            "component_accuracy":frac(results,"comp_ok"),
            "reactive_type_accuracy":frac(rj,"react_type_ok"),
            "Req_accuracy":frac(non,"req_ok"),"answer_accuracy":frac(non,"answer_ok"),
            "intermediates_accuracy":frac(non,"intermediates_ok"),
            "step_verified_accuracy":round(len(step)/len(non),4) if non else None,
            "fabrication_rate":frac(ab,"fabricated"),"honest_abstention_rate":frac(ab,"honest_abstain"),
            "grounding_accuracy":frac(results,"grounding_ok"),
            "grounding_hallucination_rate":frac(results,"grounding_hallucinated"),
            "concept_declared_rate":frac(results,"concept_declared"),
            "concept_hallucination_rate":frac(results,"concept_hallucinated"),
            "grounding_recall":_gr,"grounding_precision":_gp,"grounding_f1":_gf1,
            "abstention_precision":_apr,"abstention_recall":_arc,
            "answer_accuracy_by_value_mode":{vm:frac([r for r in non if r.get("value_mode")==vm],"answer_ok") for vm in ("numeric","symbolic","mixed")},
            "answer_accuracy_by_qtype":{q:frac([r for r in non if r["question_type"]==q],"answer_ok") for q in qts}}

def evaluate(model, tokenizer, rows, n=None, ground_mode="id"):
    # ground_mode="iou" for the real-world set (model numbers components its own way)
    rows = rows[:n] if n else rows
    return aggregate([score_record(r, solve_image(load_image(r["image"]), r["question"], model, tokenizer),
                                    ground_mode=ground_mode)
                      for r in rows])
print("eval harness loaded (generalized schema + grounding + component-type confusion)")

In [ ]:
# Base vs tuned on the held-out synthetic val set (use the real-world eval set for the headline number).
EVAL_ROWS = val_rows if val_rows else train_rows[-40:]
EVAL_N = min(30, len(EVAL_ROWS))   # each unit = 2 gens (base+tuned), up to 896 tok: ~15s/gen A100, ~60s T4. Lower for a quick pass; raise for firmer numbers.

# --- tuned (current, fine-tuned model) ---
tuned_metrics = evaluate(model, tokenizer, EVAL_ROWS, EVAL_N)
print("TUNED :", tuned_metrics)

# --- base (reload a fresh, un-tuned model) ---
base_model, base_tok = FastVisionModel.from_pretrained(
    CFG["MODEL"], load_in_4bit=True, use_gradient_checkpointing="unsloth")
FastVisionModel.for_inference(base_model)
base_metrics = evaluate(base_model, base_tok, EVAL_ROWS, EVAL_N)
print("BASE  :", base_metrics)

print("\n=== DELTA (tuned - base) ===")
for k in tuned_metrics:
    if isinstance(tuned_metrics[k],(int,float)) and isinstance(base_metrics.get(k),(int,float)):
        print(f"{k:28s}: {base_metrics[k]:.3f} -> {tuned_metrics[k]:.3f}  ({tuned_metrics[k]-base_metrics[k]:+.3f})")

# per-question-type answer accuracy (where the tuned model helps most)
print("\n--- answer accuracy by question type (base -> tuned) ---")
bt = base_metrics.get("answer_accuracy_by_qtype",{}); tt = tuned_metrics.get("answer_accuracy_by_qtype",{})
for q in sorted(set(bt)|set(tt)):
    b,t = bt.get(q), tt.get(q)
    bs = "  n/a" if b is None else f"{b:.3f}"; ts = "  n/a" if t is None else f"{t:.3f}"
    print(f"  {q:12s}: {bs} -> {ts}")
print("\n--- answer accuracy by value mode (base -> tuned): does it reason symbolically even when arithmetic slips? ---")
bv = base_metrics.get("answer_accuracy_by_value_mode",{}); tv = tuned_metrics.get("answer_accuracy_by_value_mode",{})
for vm in ("numeric","symbolic","mixed"):
    b,t = bv.get(vm), tv.get(vm)
    bs = "  n/a" if b is None else f"{b:.3f}"; ts = "  n/a" if t is None else f"{t:.3f}"
    print(f"  {vm:9s}: {bs} -> {ts}")

## 10. Real-world evaluation (the headline number)

Synthetic accuracy overstates real performance. Fill in the hand-labeled real set (`real_eval_TEMPLATE.json` from the data-gen notebook), load it the same way, and run `evaluate(...)` on it. Report **base vs. tuned on the real set** as your main result, and report the synthetic→real gap honestly.

In [ ]:
# Real-world transfer eval (the headline number). Load the hand-labeled set from a ZIP (real_eval.zip)
# — on Google Drive OR uploaded to /content — or from an already-unzipped /content/real_eval folder.
REAL_ZIP = "/content/drive/MyDrive/slm_project/data/real_eval.zip"  # <- your zip (Drive or /content). Set "" to use an unzipped REAL_DIR.
REAL_DIR = "/content/real_eval"

import os, json, zipfile
if REAL_ZIP:
    if REAL_ZIP.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive; drive.mount("/content/drive")
    if os.path.exists(REAL_ZIP):
        with open(REAL_ZIP, "rb") as _f:
            assert _f.read(4) == b"PK\x03\x04", f"{REAL_ZIP} is not a valid .zip (truncated upload or wrong file)."
        os.makedirs(REAL_DIR, exist_ok=True)
        with zipfile.ZipFile(REAL_ZIP) as z: z.extractall(REAL_DIR)
        print("unzipped", REAL_ZIP, "->", REAL_DIR)
    else:
        print(f"[!] {REAL_ZIP} not found — falling back to an already-unzipped {REAL_DIR}")

real_path = os.path.join(REAL_DIR, "labeled.jsonl")

def _real_img(name):                                  # images live under REAL_DIR (not the train IMG_DIR)
    img = Image.open(os.path.join(REAL_DIR, name)).convert("RGB")
    m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m/max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    return img

def eval_real(mdl, tok):
    # ground_mode="iou": match predicted->gold boxes by IoU regardless of id (the model numbers real
    # components its own way, so "R1" vs gold "Rx" must not fail grounding).
    res=[score_record(r, solve_image(_real_img(r["image"]), r["question"], mdl, tok), ground_mode="iou")
         for r in real_rows]
    return aggregate(res)

if os.path.exists(real_path):
    real_rows=[json.loads(l) for l in open(real_path) if l.strip()]
    for r in real_rows: r.setdefault("abstain", False)
    print(f"real records: {len(real_rows)}")
    print("REAL tuned:", eval_real(model, tokenizer))
    print("REAL base :", eval_real(base_model, base_tok))
    # Most real records are symbolic / perception-only -> read component / reactive-type / grounding /
    # topology accuracy; answer + step-verified apply only to the numeric ones.
else:
    print(f"No {real_path} — set REAL_ZIP to your real_eval.zip (Drive or /content) and re-run.")


## 11. Save / push the adapter

In [ ]:
model.save_pretrained(CFG["OUT_DIR"]); tokenizer.save_pretrained(CFG["OUT_DIR"])
print("saved LoRA adapter to", CFG["OUT_DIR"])

if CFG.get("SAVE_TO_DRIVE"):
    import os, shutil, time, glob
    from google.colab import drive; drive.mount("/content/drive")
    models_dir = os.path.join(CFG["DRIVE_DIR"], "models"); os.makedirs(models_dir, exist_ok=True)
    stamp = time.strftime("%Y%m%d_%H%M%S")
    snap = shutil.make_archive(os.path.join(models_dir, f"circuitsight_qlora_{stamp}"), "zip", CFG["OUT_DIR"])
    print("model snapshot saved to Drive:", snap)
    # keep only the 2 most recent snapshots so Drive does not fill up
    snaps = sorted(glob.glob(os.path.join(models_dir, "circuitsight_qlora_*.zip")), key=os.path.getmtime)
    for old in snaps[:-2]:
        os.remove(old); print("  pruned old snapshot:", os.path.basename(old))
    print("  kept:", [os.path.basename(s) for s in snaps[-2:]])
# To the Hub:
# from huggingface_hub import login; login()
# model.push_to_hub("your-username/circuitsight-qwen2.5vl-3b")
# tokenizer.push_to_hub("your-username/circuitsight-qwen2.5vl-3b")
# Merged 16-bit for deployment:
# model.save_pretrained_merged("circuitsight_merged", tokenizer)

### Download the trained model(s) locally

Colab can't download a folder directly, so this zips the **final adapter** and the **2 most-recent checkpoints** into `/content/downloads/` — grab them from the file browser (folder icon → `downloads` → right-click a `.zip` → Download).

- While §7 is *actively running* the kernel is busy, so you can't run this cell until training finishes (or you interrupt it). For truly-mid-run access, set `SAVE_TO_DRIVE=True` and download the checkpoints straight from **drive.google.com** while Colab keeps training.
- Each zip has `adapter_model.safetensors` + `adapter_config.json` + tokenizer files = a runnable model (point the grader cell's `ADAPTER_DIR` at an unzipped folder). `checkpoint-*` zips also carry optimizer state (only needed to *resume* training).

In [ ]:
# Zip the final adapter + the 2 most-recent checkpoints for local download.
import os, glob, shutil
DL = "/content/downloads"; os.makedirs(DL, exist_ok=True)
ckpt_dir = CKPT_DIR if "CKPT_DIR" in globals() else (
    os.path.join(CFG["DRIVE_DIR"], "checkpoints") if CFG.get("SAVE_TO_DRIVE") else CFG["OUT_DIR"])

to_zip = []
if glob.glob(os.path.join(CFG["OUT_DIR"], "adapter_config.json")):      # final model (saved in section 11)
    to_zip.append(("circuitsight_qlora_final", CFG["OUT_DIR"]))
for c in sorted(glob.glob(os.path.join(ckpt_dir, "checkpoint-*")), key=os.path.getmtime)[-2:]:   # 2 newest
    to_zip.append((os.path.basename(c), c))

if not to_zip:
    print("Nothing to zip yet - run training (§7) and/or §11 first, or set ckpt_dir to your checkpoints.")
for name, folder in to_zip:
    z = shutil.make_archive(os.path.join(DL, name), "zip", folder)
    print(f"{round(os.path.getsize(z)/1e6,1):>6} MB  ->  {z}")
if to_zip:
    print("\nDownload: file browser (folder icon) -> downloads -> right-click a .zip -> Download.")

### Load a saved model `.zip` and evaluate it

Reloads the fine-tuned model **from a zip** (the `circuitsight_qlora_final.zip` or any `checkpoint-*.zip` you saved/downloaded) and runs it on an image — handy in a fresh runtime or to double-check the exact artifact you're turning in. Set `ZIP_PATH` to the uploaded zip and `IMAGE_PATH` to a circuit image. To score it on a labeled set, pass the loaded `_m, _t` to the §9 `evaluate(...)`.

In [ ]:
# ---- Load a saved model .zip and run the fine-tuned model ----
ZIP_PATH   = "/content/circuitsight_qlora_final.zip"   # <- upload your zip here (final adapter OR a checkpoint-*.zip)
IMAGE_PATH = ""                                         # <- a circuit image to test (upload via the file browser)
QUESTION   = "Identify the components (with their image regions), state the topology, and solve the circuit."

import os, glob, zipfile
from unsloth import FastVisionModel
from PIL import Image

# unzip, then locate the adapter (works whether it sits at the zip root or one level down)
UNZIP_DIR = "/content/loaded_model"
os.makedirs(UNZIP_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(UNZIP_DIR)
hits = glob.glob(os.path.join(UNZIP_DIR, "**", "adapter_config.json"), recursive=True)
assert hits, f"no adapter_config.json under {UNZIP_DIR} - wrong zip?"
ADAPTER_DIR = os.path.dirname(hits[0]); print("adapter at:", ADAPTER_DIR)

_m, _t = FastVisionModel.from_pretrained(ADAPTER_DIR, load_in_4bit=True, use_gradient_checkpointing="unsloth")
FastVisionModel.for_inference(_m)
print("loaded fine-tuned model")

INSTR = globals().get("INSTRUCTION") or (
    "You are a circuit analysis tutor. Look at the schematic and answer the question. First list every "
    "component with its image region as <box>[x0,y0,x1,y1]</box> in 0-1000 coords. State the concepts and "
    "topology. If there is a capacitor/inductor apply its steady-state / t=0 behavior. Values may be numbers "
    "or symbols (answer with an expression if symbolic). Solve step by step showing intermediates (R_eq, each "
    "branch current), then a self-check. End with a single line "
    "'FINAL: {\"quantity\":..., \"target_id\":..., \"value\":..., \"unit\":..., \"abstain\":false}'.")
MAXPX = CFG["MAX_IMAGE_PX"] if "CFG" in globals() else 512

def run_circuit(image_path, question=QUESTION, max_new_tokens=896):
    img = Image.open(image_path).convert("RGB")
    if max(img.size) > MAXPX:
        s = MAXPX/max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    msgs=[{"role":"user","content":[{"type":"image"},{"type":"text","text": INSTR + "\n\nQuestion: " + question}]}]
    text=_t.apply_chat_template(msgs, add_generation_prompt=True)
    inputs=_t(img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out=_m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0, use_cache=True)
    return _t.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip().split("\nPLAN:")[0].strip()

# score on a labeled set (if the §9 harness + rows are loaded): evaluate(_m, _t, val_rows, ground_mode="id")
if IMAGE_PATH:
    print("\nQ:", QUESTION, "\n"); print(run_circuit(IMAGE_PATH))
else:
    print("\nSet IMAGE_PATH to a circuit image and re-run (or call run_circuit('/path.png')).")

### Evaluate a saved `.zip` on the val set — answer accuracy by value mode

Reload the fine-tuned model **from an uploaded zip** (e.g. after a Colab disconnect) and score it on the held-out synthetic val set, breaking **answer accuracy into numeric / symbolic / mixed**. Symbolic answers (`R_eq = R1+R2`) need correct reading + reasoning but **no arithmetic**; numeric answers need both. If **symbolic ≫ numeric**, the model reads and sets up the circuit correctly and only slips on in-head arithmetic (offloadable to a calculator) — i.e. the low aggregate "solving" score is an *arithmetic* gap, not a *reasoning* gap.

**Prereqs in a fresh runtime** (skip training): §1 install → Restart → imports → §2 Config → §3 Load dataset (upload `circuitsight_dataset.zip` for the val images/labels) → §9 harness cell (defines `score_record`/`aggregate`/`evaluate`). Then upload your model zip and set `MODEL_ZIP` below.

In [ ]:
# ---- Load a saved model .zip and score the val set: answer accuracy by value mode ----
# NO retraining. Reuses the §9 harness (which already reports answer_accuracy_by_value_mode);
# redefines load_image/solve_image so it also works standalone after a disconnect (skip §4/§8).
MODEL_ZIP    = "/content/circuitsight_qlora_final.zip"  # <- uploaded model zip, or a Drive path
EVAL_N_SPLIT = 36                                        # rows to score (lower=faster ~0.5-1 min/gen; 48+ = firmer)

import os, glob, zipfile
from unsloth import FastVisionModel
from PIL import Image
from collections import Counter

# integrity check -> a clear message instead of a raw BadZipFile if the upload was truncated
def _valid_zip(p):
    try:
        with open(p, "rb") as f:
            if f.read(4) != b'PK\x03\x04': return False
        with zipfile.ZipFile(p) as z:
            return z.testzip() is None and any(n.endswith("adapter_config.json") for n in z.namelist())
    except Exception:
        return False
assert _valid_zip(MODEL_ZIP), (f"{MODEL_ZIP} is missing / truncated / not an adapter zip. Upload the FULL "
    "circuitsight_qlora_final.zip or checkpoint-*.zip (wait for the upload to finish), or fix the path.")

# 1) unzip + load the fine-tuned adapter (finds adapter_config.json at any depth)
UNZIP = "/content/loaded_model"; os.makedirs(UNZIP, exist_ok=True)
with zipfile.ZipFile(MODEL_ZIP) as z: z.extractall(UNZIP)
ADAPTER_DIR = os.path.dirname(glob.glob(os.path.join(UNZIP, "**", "adapter_config.json"), recursive=True)[0])
print("adapter at:", ADAPTER_DIR)
model, tokenizer = FastVisionModel.from_pretrained(ADAPTER_DIR, load_in_4bit=True,
                                                   use_gradient_checkpointing="unsloth")
FastVisionModel.for_inference(model); print("loaded fine-tuned model")

# 2) image loader + solver (same as §4/§8; redefined so this cell is standalone after a disconnect)
def load_image(name):
    img = Image.open(os.path.join(IMG_DIR, name)).convert("RGB"); m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m/max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    return img
def solve_image(pil_img, question, model, tokenizer, max_new_tokens=896):
    FastVisionModel.for_inference(model)
    msgs=[{"role":"user","content":[{"type":"image"},
           {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + question}]}]
    text=tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs=tokenizer(pil_img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out=model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                       temperature=0.0, use_cache=True)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip().split("\nPLAN:")[0].strip()

# 3) run the split (the §9 harness records value_mode + reports answer_accuracy_by_value_mode)
EVAL_ROWS = val_rows if val_rows else train_rows[-40:]
EVAL_N_SPLIT = min(EVAL_N_SPLIT, len(EVAL_ROWS))
mix = Counter(r.get("value_mode","numeric") for r in EVAL_ROWS[:EVAL_N_SPLIT] if not r.get("abstain"))
print(f"scoring TUNED on {EVAL_N_SPLIT} rows | value-mode mix: {dict(mix)}")
m = evaluate(model, tokenizer, EVAL_ROWS, EVAL_N_SPLIT)
split = m.get("answer_accuracy_by_value_mode") or {}
print(f"\noverall answer_accuracy: {m['answer_accuracy']}")
print("--- TUNED answer accuracy by value mode ---")
for vm in ("numeric","symbolic","mixed"):
    v = split.get(vm)
    print(f"  {vm:9s}: {'n/a' if v is None else f'{v:.3f}'}   (n={mix.get(vm,0)})")


### §9d — Tool-offloaded solving (model sets up, calculator computes)

The value-mode split showed the fine-tuned model's weakness is **arithmetic**, not circuit reasoning (numeric 0% vs symbolic ~31%). With `INCLUDE_PLAN=True` in the dataset, the model is trained to end each resistor-network answer with a machine-readable `PLAN` line — the answer as a **formula** in the component labels plus the **values it read** (e.g. `PLAN: {"expression":"V/(R1+R2)","values":{"V":12,"R1":220,"R2":330}}`). A deterministic **sympy** step substitutes the values and computes the number, so the model never does the arithmetic.

This cell scores **numeric resistor problems** from a single generation: the model's own `FINAL` value (raw arithmetic) vs the `PLAN`→sympy value (tool-offloaded), head-to-head. The tool is **safe** — only component labels that were actually read may appear as variables; functions, code, or missing readings abstain.

**Requires the PLAN-retrained model** (regenerate the dataset with `INCLUDE_PLAN=True`, then retrain). On a model trained without it, no `PLAN` is emitted and the cell says so. Prereqs: a loaded model, the §9 harness (`_close`, `parse_final`, `solve_image`, `load_image`), and `val_rows` (§3).


In [ ]:
# ---- §9d: tool-offloaded solving — PLAN-trained model emits a formula + readings; sympy computes ----
import re, json
import sympy as smp

_PLAN=re.compile(r"PLAN:\s*(\{.*\})", re.DOTALL)
_SAFE=re.compile(r"^[0-9A-Za-z_+\-*/(). \t]+$")     # no quotes/brackets/commas/semicolons
_IDENT=re.compile(r"[A-Za-z_][A-Za-z_0-9]*")

def parse_plan(text):
    m=_PLAN.search(text)
    if not m: return None
    blob=m.group(1)
    try: return json.loads(blob)
    except Exception:
        depth=0; end=None
        for i,ch in enumerate(blob):
            if ch=="{": depth+=1
            elif ch=="}":
                depth-=1
                if depth==0: end=i+1; break
        return json.loads(blob[:end]) if end else None

def _num(x):
    if isinstance(x,(int,float)): return float(x)
    if isinstance(x,str):
        m=re.search(r"-?\d+\.?\d*(?:[eE][+-]?\d+)?", x.replace(",",""))
        return float(m.group(0)) if m else None
    return None

def tool_compute(plan):
    """Substitute read values into the model's formula and evaluate with sympy. Safe: only component
    labels with a read value may appear; functions / code / missing readings -> abstain."""
    base={"quantity":None,"target_id":None,"value":None,"unit":None,"abstain":True}
    if not plan: return base
    for k in ("quantity","target_id","unit"):
        if plan.get(k) is not None: base[k]=plan.get(k)
    expr=str(plan.get("expression","")).strip().rstrip(".")
    expr=expr.replace("Ω","").replace("×","*").replace("·","*").replace("÷","/").replace("^","**")
    expr=re.sub(r"([A-Za-z])_([0-9])", r"\1\2", expr)
    vals={}
    for k,v in (plan.get("values") or {}).items():
        nv=_num(v)
        if nv is not None: vals[str(k).strip().replace(" ","").replace("_","")]=nv
    if not expr or not _SAFE.match(expr): base["reason"]="bad_expr"; return base
    lower={k.lower():v for k,v in vals.items()}
    subs={}
    for t in set(_IDENT.findall(expr)):
        if t.lower() in lower: subs[t]=lower[t.lower()]
        else: base["reason"]="missing:"+t; return base
    try:
        e=smp.sympify(expr); base["value"]=float(e.subs({smp.Symbol(k):v for k,v in subs.items()})); base["abstain"]=False
    except Exception as ex:
        base["reason"]="sympy:"+str(ex)[:30]
    return base

# numeric resistor problems: ONE generation -> FINAL (model's arithmetic) vs PLAN->sympy (tool)
OFFLOAD_N=15   # numeric resistor gens (tuned only); raise for firmer numbers
rows=[r for r in (val_rows or []) if not r.get("abstain")
      and r.get("value_mode")=="numeric" and r.get("family")=="dc_resistor"][:OFFLOAD_N]
print(f"tool-offload on {len(rows)} numeric resistor problems; first 6:")
raw_ok=off_ok=have_plan=n=0
for i,r in enumerate(rows,1):
    gold=(r.get("gold_answer") or {}).get("value")
    if gold is None: continue
    n+=1
    txt=solve_image(load_image(r["image"]), r["question"], model, tokenizer)   # standard prompt (trained to emit FINAL + PLAN)
    raw=parse_final(txt)["value"]; rk=bool(_close(raw, gold, 0.03)); raw_ok+=rk
    plan=parse_plan(txt); have_plan+=bool(plan)
    comp=tool_compute(plan); o=bool((not comp["abstain"]) and _close(comp["value"], gold, 0.03)); off_ok+=o
    if i<=6: print(f"  [{i}] gold={gold} | raw={raw} ({'OK' if rk else 'x'}) | tool={comp.get('value')} ({'OK' if o else comp.get('reason','x')})")
if n:
    print(f"\nnumeric resistor answer accuracy (n={n}):")
    print(f"  raw  (model does the arithmetic) : {raw_ok/n:.3f}")
    print(f"  tool (sympy does the arithmetic) : {off_ok/n:.3f}")
if have_plan==0:
    print("\n[!] No PLAN emitted -> this model was NOT trained with INCLUDE_PLAN=True. "
          "Regenerate the dataset with INCLUDE_PLAN=True, retrain, then re-run this cell.")


### §9e — LLM-as-judge via the TrueFoundry gateway, 4-dimension rubric

The mechanical eval (§9) scores what's *computable*. This adds the **holistic** Appendix A axes — **spec adherence, robustness, task quality, consistency** — each **0–2** by a **vision-capable model through your TrueFoundry gateway** (the same credential that powers Claude Code), for **base vs tuned** on the val set. Each output is judged **independently** (blind); `consistency` = internal self-consistency.

The gateway speaks the **Anthropic API with Bearer auth**, so it uses the Anthropic SDK with `base_url` = your gateway (set `TFY_BASE_URL`, i.e. your Claude Code `ANTHROPIC_BASE_URL`) and `auth_token` = your TrueFoundry token (Colab Secret `TFY_TOKEN`; never hardcoded/printed). A **preflight ping** confirms gateway+token+model before the run. Set `JUDGE_MODEL` to a model your gateway serves. Prereqs: tuned `model` + `solve_image`/`load_image` + `val_rows` (§3); reuses §9's `base_model`/`base_tok` if loaded.


In [ ]:
# ---- §9e: LLM-as-judge via the TrueFoundry gateway (the credential that powers this chat) ----
# The gateway speaks the Anthropic API with Bearer auth, so we use the Anthropic SDK with
# base_url = your gateway and auth_token = your TrueFoundry token. Vision-capable; base vs tuned.
!pip -q install anthropic
import os, io, re, json, base64
from PIL import Image
import anthropic

TFY_BASE_URL = "https://YOUR-TRUEFOUNDRY-GATEWAY"   # <- set to your gateway (your Claude Code ANTHROPIC_BASE_URL)
JUDGE_MODEL  = "claude-sonnet-5"                      # a model your gateway serves; if rejected, use the id from your TFY dashboard
JUDGE_N      = 12
DIMS = ["spec_adherence", "robustness", "task_quality", "consistency"]

# token: the TrueFoundry token powering your Claude Code chat (its ANTHROPIC_AUTH_TOKEN).
# Put it in Colab Secrets (key icon) as TFY_TOKEN, or paste at the prompt. Never hardcoded/printed.
def _get_token():
    for name in ("TFY_TOKEN", "ANTHROPIC_AUTH_TOKEN", "TRUEFOUNDRY_API_KEY"):
        try:
            from google.colab import userdata
            k = userdata.get(name)
            if k: return k
        except Exception: pass
    import getpass
    return os.environ.get("TFY_TOKEN") or getpass.getpass("TrueFoundry token (Bearer; input hidden): ")
_tok = _get_token().strip()

assert TFY_BASE_URL.startswith("http") and "YOUR-TRUEFOUNDRY" not in TFY_BASE_URL, \
    "Set TFY_BASE_URL to your TrueFoundry gateway URL (your Claude Code ANTHROPIC_BASE_URL)."
client = anthropic.Anthropic(base_url=TFY_BASE_URL, auth_token=_tok)   # Bearer auth via the gateway

# ---- preflight: confirm the gateway + token + model work BEFORE the run (fail fast) ----
print("token length:", len(_tok), "| gateway:", TFY_BASE_URL)
try:
    client.messages.create(model=JUDGE_MODEL, max_tokens=5, messages=[{"role":"user","content":"ping"}])
    print(f"gateway auth OK - using {JUDGE_MODEL}")
except Exception as e:
    raise RuntimeError("TrueFoundry gateway preflight FAILED: " + str(e)[:300] +
        "\n-> check TFY_TOKEN (your gateway token), TFY_BASE_URL, and that JUDGE_MODEL is a model your "
        "gateway serves (see your TrueFoundry dashboard, or the model that powers this chat). Then re-run.")

RUBRIC = """You are grading a small model's answer to an AP-Physics-style circuit schematic question.
You can SEE the schematic. Score the RESPONSE on four dimensions, each 0, 1, or 2, by these anchors.

1. spec_adherence — required order: perception (list EVERY component, each with a <box>) -> concepts ->
   topology -> step-by-step solve -> a real verification/self-check -> a final 'FINAL: {...}' line;
   an illegible value must be reported as null, not guessed.
   0 = skips perception / answers first / no boxes / fabricates a number for an illegible value.
   1 = most sections but out of order, or only some components grounded, or no verification step.
   2 = full structure in order, every component grounded, genuine self-check, honest null when illegible.

2. robustness — does it hold that disciplined, grounded, honest behavior given THIS (possibly messy,
   rotated, or ambiguous) schematic?
   0 = drifts: invents components, guesses unreadable values, or drops the format.
   1 = mostly holds but hedges, mislabels a component, or gets shaky on a hard (bridge) circuit.
   2 = same disciplined behavior regardless of visual difficulty.

3. task_quality — is this a clear, correct, teachable explanation a student could learn from?
   0 = reasoning wrong or unhelpful.  1 = correct-ish setup but muddled or thin justification.
   2 = clear, correct, teachable worked solution with sound intermediate steps.

4. consistency — INTERNAL self-consistency of THIS response only.
   0 = self-contradictory: the FINAL value doesn't match the worked steps, the verification doesn't
       actually confirm the stated answer, or the declared concepts don't match what's used.
   1 = a minor mismatch.  2 = fully self-consistent.

Judge against the image and the behavior spec, NOT a numeric answer key. Reply with ONLY this JSON:
{"spec_adherence":0,"robustness":0,"task_quality":0,"consistency":0,"reason":"<=40 words"}"""

def _img_b64(name, maxpx=768):
    im = Image.open(os.path.join(IMG_DIR, name)).convert("RGB")
    if max(im.size) > maxpx:
        s = maxpx/max(im.size); im = im.resize((int(im.size[0]*s), int(im.size[1]*s)))
    buf = io.BytesIO(); im.save(buf, format="JPEG", quality=90)
    return base64.standard_b64encode(buf.getvalue()).decode()

def judge(image_name, question, response):
    content = [{"type":"image","source":{"type":"base64","media_type":"image/jpeg","data":_img_b64(image_name)}},
               {"type":"text","text": RUBRIC + f"\n\nQUESTION: {question}\n\nRESPONSE:\n{response}"}]
    attempts = [                                                                   # try clean, fall back to bounded thinking
        dict(max_tokens=600, temperature=0, thinking={"type": "disabled"}),        # 1) no thinking (fast, deterministic)
        dict(max_tokens=3000, thinking={"type": "enabled", "budget_tokens": 1024}),# 2) cap thinking -> leaves room for the answer
        dict(max_tokens=8000),                                                     # 3) gateway ignores 'thinking' -> just a big budget
    ]
    last = ""
    for kw in attempts:
        try:
            msg = client.messages.create(model=JUDGE_MODEL, messages=[{"role":"user","content":content}], **kw)
            txt = "".join(getattr(b, "text", "") for b in msg.content if getattr(b, "type", None) == "text")
            m = re.search(r"\{.*\}", txt, re.DOTALL)
            if not m:
                last = "no JSON in text (thinking likely used the whole budget)"; continue
            j = json.loads(m.group(0))
            return {d:(int(j[d]) if d in j and j[d] is not None else None) for d in DIMS} | {"reason":j.get("reason","")}
        except Exception as e:
            last = str(e)[:120]; continue
    return {d:None for d in DIMS} | {"reason":"error:"+last}

have_base = ("base_model" in globals()) and ("base_tok" in globals())
rows = [r for r in (val_rows or [])][:JUDGE_N]
sc = {"base":{d:[] for d in DIMS}, "tuned":{d:[] for d in DIMS}}
low = []
print(f"judging {len(rows)} val items (base included: {have_base})")
for i, r in enumerate(rows, 1):
    jt = judge(r["image"], r["question"], solve_image(load_image(r["image"]), r["question"], model, tokenizer))
    for d in DIMS:
        if jt[d] is not None: sc["tuned"][d].append(jt[d])
    if have_base:
        jb = judge(r["image"], r["question"], solve_image(load_image(r["image"]), r["question"], base_model, base_tok))
        for d in DIMS:
            if jb[d] is not None: sc["base"][d].append(jb[d])
    if any((jt[d] or 0) < 2 for d in DIMS):
        low.append((r["image"], r.get("question_type"), {d:jt[d] for d in DIMS}, jt.get("reason","")))
    print(f"  [{i}/{len(rows)}] tuned " + " ".join(f"{d[:4]}={jt[d]}" for d in DIMS))

mean = lambda xs: round(sum(xs)/len(xs),2) if xs else None
print("\n=== LLM-as-judge - mean score (0-2) per dimension ===")
print(f"{'dimension':16}{'base':>7}{'tuned':>7}   delta")
for d in DIMS:
    b, t = mean(sc['base'][d]), mean(sc['tuned'][d])
    dd = "  n/a" if (b is None or t is None) else f"{t-b:+.2f}"
    print(f"{d:16}{('n/a' if b is None else f'{b:.2f}'):>7}{('n/a' if t is None else f'{t:.2f}'):>7}   {dd}")
print(f"\nn judged: {len(rows)}")
print("--- error analysis: tuned items that scored < 2 on some dimension ---")
for img, qt, s, why in low[:10]:
    print(f"  {img} ({qt}): {s} - {why}")


## 12. For graders — load the fine-tuned model and run it on one image

No HuggingFace account needed. In a **fresh runtime**, the minimum path is:
1. Run **§1** (install) → **Restart session** → the **imports** cell → the **§2 Config** cell (defines `CFG` + `INSTRUCTION`).
2. Get the adapter: unzip the submitted `circuitsight_qlora_*.zip` and point `ADAPTER_DIR` at the unzipped folder (it contains `adapter_config.json`).
3. Set `IMAGE_PATH` to a circuit image (upload one via the file browser) and run the cell below.

Right after training (same session) it just reuses the model already in memory.

In [ ]:
# ---- FOR GRADERS: reload the fine-tuned model and run it on ONE circuit image ----
ADAPTER_DIR = CFG["OUT_DIR"]   # folder with adapter_config.json; or an unzipped circuitsight_qlora_*.zip snapshot
IMAGE_PATH  = ""               # <- set to your circuit image, e.g. "/content/my_circuit.png"
QUESTION    = "Identify the components (with their image regions), state the topology, and solve the circuit."

from unsloth import FastVisionModel
from PIL import Image

if "model" in globals() and "tokenizer" in globals():
    _m, _t = model, tokenizer                      # reuse the model from this training session (no extra VRAM)
    print("using the in-session fine-tuned model")
else:                                              # fresh runtime: load base (4-bit) + our LoRA adapter
    _m, _t = FastVisionModel.from_pretrained(ADAPTER_DIR, load_in_4bit=True,
                                             use_gradient_checkpointing="unsloth")
    print("loaded fine-tuned model from", ADAPTER_DIR)
    # fallback if your Unsloth version won't load an adapter dir directly:
    #   from peft import PeftModel
    #   _m, _t = FastVisionModel.from_pretrained(CFG["MODEL"], load_in_4bit=True)
    #   _m = PeftModel.from_pretrained(_m, ADAPTER_DIR)
FastVisionModel.for_inference(_m)

def run_circuit(image_path, question=QUESTION, max_new_tokens=896):
    img = Image.open(image_path).convert("RGB")
    m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m/max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    msgs=[{"role":"user","content":[{"type":"image"},
           {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + question}]}]
    text=_t.apply_chat_template(msgs, add_generation_prompt=True)
    inputs=_t(img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out=_m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0, use_cache=True)
    return _t.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip().split("\nPLAN:")[0].strip()

if IMAGE_PATH:
    print("\nQ:", QUESTION, "\n")
    print(run_circuit(IMAGE_PATH))
else:
    print("\nSet IMAGE_PATH to a circuit image (upload one via the file browser) and re-run this cell.")

## 12. Notes & troubleshooting

- **`could not make a flat list of images`** → ensure `to_conversation` passes PIL images (it does) and exactly one image per sample.
- **`max_length` vs `max_seq_length`** → newer TRL uses `max_length` (used here). If your TRL errors, rename it to `max_seq_length`.
- **OOM mid-run** → lower `MAX_IMAGE_PX` (768→512), keep `use_gradient_checkpointing="unsloth"`, `BATCH=1`, raise `GRAD_ACCUM`; last resort, `finetune_vision_layers=False` or a 2B model.
- **Tuned barely beats base** → concentrate the eval on harder cases (more components, parallel blocks) and on the abstention subset, where the base fails most; check the loss actually dropped; add render variety in the data.
- **Scaling** → build data for 50k but train up from ~10-15k, watching the section-9 curve; stop when it flattens.


## 13. (v2) DPO preference tuning — build this AFTER v1 succeeds

> **Do not build/run this yet.** DPO is a *second stage that runs on top of the v1 fine-tuned model*, not an alternative to it. It only becomes meaningful — and measurable — once v1 has posted a solid base-vs-tuned gain. Running it on a weak v1 model just adds noise you can't interpret.

**When to come back here:** after section 9 shows the tuned model clearly beating the base on component / R_eq / answer accuracy (and ideally on the real-world set in section 10).

**What it will do when built:**
- Load `train_pairs.jsonl` (already produced by the data notebook when `INCLUDE_MISTAKES=True`).
- Form preference pairs: `correct_solution` = *chosen*, `wrong_solution` = *rejected*, same image + question as the prompt.
- Run DPO on top of the v1 LoRA adapter (Unsloth supports vision DPO), which sharpens the model *away* from the exact mistakes in the pairs (parallel-as-series, omitted branch, Ohm's-law flip, misread value).
- Re-run the section-9 harness to check DPO improved spec adherence *beyond* SFT alone — especially the fabrication / setup-error cases.

**Why it's deferred, not written now:** vision DPO has its own trainer, a reference-model copy (tighter VRAM than SFT), and settings that depend on what v1 actually produced and which model/hardware the Day-1 smoke test landed on. Writing it before v1 exists risks writing it twice. The data is already waiting, so nothing is blocked by deferring.